# Workshop 1 — Basics

This notebook walks you through connecting to the Menlo Robotics platform, creating a simulated robot, and sending it movement commands using the **menlo_robot_sdk**.

By the end of this tutorial you will:
1. Authenticate with the Menlo platform using an API key
2. Create a virtual robot and connect to it
3. Open a 3D viewer to watch the robot in your browser
4. Discover the robot's available skills
5. Drive the robot with `set_velocity` and `go_to` commands
6. Capture the robot's point-of-view camera image
7. Write a simple chain of commands to navigate around obstacles

## Prerequisites

Before you start, make sure you have:

- A **Google account** (to use Google Colab)
- A Menlo API key (get one from [platform.menlo.ai](http://platform.menlo.ai/) -> Settings -> API Keys)
- **Google Chrome** (the 3D viewer works best in Chrome)

## Setting Up Your API Keys

This notebook supports two modes — pick the one that fits your environment:

### Mode A: Google Colab (Secrets manager)

Colab's built-in Secrets manager stores keys securely and persists them across sessions.

1. Click the **key icon** (🔑) in the left sidebar of Colab.
2. Click **"Add new secret"** and add each key below:

| Secret name | Value |
|---|---|
| `MENLO_API_KEY` | Your Menlo API key (from [platform.menlo.ai](http://platform.menlo.ai/) → Settings → API Keys) |
| `TOKAMAK_API_KEY` | Your Tokamak API key (used in Section 7 for the VLM) |

3. Toggle **"Notebook access"** to ON for each secret.

> **Note:** Secrets are never included when you share the notebook. Each person needs to set up their own.

### Mode B: Local / `.env` file

Running locally (VS Code, JupyterLab, etc.)? Create a `.env` file in the same directory as this notebook:

```
MENLO_API_KEY=sk_live_abc123...
TOKAMAK_API_KEY=tok_abc123...
```

The next cell auto-detects which mode you're in and loads the keys accordingly.

---

# Section 4: Basic Usage of the SDK

## Key Imports from the SDK

The `menlo_robot_sdk` package provides a few core building blocks:

| Import | What it does |
|---|---|
| `AsyncClient` | HTTP client that talks to the Menlo platform (create robots, manage sessions, etc.) |
| `connect` | High-level function that opens a real-time session to a robot and returns a `MenloSession` |
| `generate_room_key` | *(experimental)* Generates a one-time key for the browser-based 3D viewer |

Once connected, the **`session`** object is your main handle:
- `session.discover_skills()` — list what the robot can do
- `session.invoke(skill, params)` — run a skill and wait for the result
- `session.get_vision(camera)` — grab a camera frame
- `session.state.get(key)` — read the robot's current state
- `session.disconnect()` — close the connection

The **`robot_id`** is the unique identifier for your robot on the platform. You get it back when you call `client.robots.create()`.

## Step 1: Install Dependencies

Colab doesn't have `menlo_robot_sdk` pre-installed, so we install it here. This cell needs to run once per session (packages don't persist when the Colab runtime resets).

In [1]:
!pip install -q "menlo-robot-sdk[livekit]"

'C:\Python314\Scripts\pip.exe' ������ Device Guard ��å�� ���� ���ܵǾ����ϴ�.
�ڼ��� ������ ���� ����ڿ��� �����ϼ���.


## Step 2: Load Configuration

We load the API key from Colab's Secrets manager using `google.colab.userdata`.

In [ ]:
import os
import time
import asyncio

# ── Key loader: tries .env first, then Colab Secrets ────────────────────────
def _load_keys():
    # Mode B: local .env file
    try:
        from dotenv import load_dotenv
        load_dotenv(override=False)
        if os.environ.get("MENLO_API_KEY"):
            print("Loaded keys from .env file")
            return "dotenv"
    except ImportError:
        pass  # python-dotenv not installed → not in .env mode

    # Mode A: Google Colab Secrets
    try:
        from google.colab import userdata
        os.environ.setdefault("MENLO_API_KEY",    userdata.get("MENLO_API_KEY") or "")
        os.environ.setdefault("TOKAMAK_API_KEY",  userdata.get("TOKAMAK_API_KEY") or "")
        print("Loaded keys from Colab Secrets")
        return "colab"
    except Exception:
        pass

    return "env"  # keys already in environment (CI, Docker, etc.)

_load_keys()

MENLO_API_KEY   = os.environ.get("MENLO_API_KEY", "")
TOKAMAK_API_KEY = os.environ.get("TOKAMAK_API_KEY", "")
RCS_URL         = "https://api.menlo.ai/rcs"
VIEWER_BASE_URL = "https://sim.menlo.ai"

assert MENLO_API_KEY and not MENLO_API_KEY.startswith("sk_live_YOUR"), (
    "MENLO_API_KEY not set!\n"
    "  • Colab: add it in the Secrets manager (key icon in the left sidebar)\n"
    "  • Local: add MENLO_API_KEY=... to a .env file next to this notebook"
)
print(f"Configuration loaded — platform: {RCS_URL}")
print(f"MENLO_API_KEY    : {MENLO_API_KEY[:12]}...")
print(f"TOKAMAK_API_KEY  : {'(set)' if TOKAMAK_API_KEY else '(not set — needed for Section 7)'}")

AssertionError: MENLO_API_KEY not set!
  • Colab: add it in the Secrets manager (key icon in the left sidebar)
  • Local: add MENLO_API_KEY=... to a .env file next to this notebook

## A Quick Primer: What Does `await` Mean?

Almost every SDK call in this tutorial starts with the keyword `await`. If you haven't seen this before, here's the 60-second version.

**The problem:** talking to a robot over the internet takes time. When we ask the platform to create a robot, or tell the robot to walk somewhere, the answer doesn't come back instantly — it can take anywhere from a few milliseconds to many seconds. Regular Python code would freeze and do nothing while it waits.

**The solution:** Python's *async* (asynchronous) functions. An async function can *pause* while it waits for a slow operation (like a network reply) instead of blocking everything. The `await` keyword means:

> *"Start this operation, pause here until it finishes, then give me the result."*

```python
# A regular function call — returns immediately:
result = some_function()

# An async call — starts the work, waits for it to complete:
result = await some_async_function()
```

**The one rule you need:** every `menlo_robot_sdk` call that talks to the platform or the robot is async, so it needs `await` in front. If you forget it, you'll get a strange value like `<coroutine object ...>` instead of your data, and Python may warn: `RuntimeWarning: coroutine was never awaited`. That warning almost always means *you forgot an `await`*.

**Why can we use `await` directly in a notebook cell?** Normally Python only allows `await` inside an `async def` function. Jupyter and Colab make an exception: notebook cells already run inside an *event loop*, so you can `await` at the top level of a cell. (If you ever move this code into a plain `.py` script, you'll need to wrap it in `async def main(): ...` and run it with `asyncio.run(main())`.)

## Step 3: Create a Client and Robot

`AsyncClient` connects to the Menlo platform using your RCS URL and API key. We then create a new simulated robot with the `asimov-v0` warehouse model.

In [ ]:
from menlo_robot_sdk import AsyncClient

client = AsyncClient(rcs_url=RCS_URL, api_key=MENLO_API_KEY)

created = await client.robots.create(name=f"sdk-test-{int(time.time())}", model="asimov-v0")
robot_id = created.robot.id
print(f"Created robot: {robot_id}")

## Step 4: Start a Session

Opening a session joins the robot's room so this notebook can send commands. The session stays open until you explicitly disconnect.

In [ ]:
from menlo_robot_sdk import connect

session = await connect(
    client,
    robot_id,
    worker_names=[],                  # no server-side worker — the browser is the runtime
    rcw_identity_prefix="simplesim",
    join_livekit=True,
)
print(f"Connected! room = {session.session.room_name}")

## Step 5: Open the 3D Viewer

The SDK generates a one-time viewer key so you can watch the robot in a browser. **Open the printed URL in a new Chrome tab** and wait for the 3D warehouse scene to fully load before continuing.

The viewer link is valid for 4 hours — you can share it like a meeting link.

In [ ]:
from menlo_robot_sdk.experimental import generate_room_key

room_key = await generate_room_key(client, robot_id)
viewer_url = f"{VIEWER_BASE_URL}/?key={room_key}"

print("=" * 60)
print(f"VIEWER URL: {viewer_url}")
print("=" * 60)
print()
print("1. Open the URL above in a new Chrome tab")
print("2. Wait for the 3D warehouse robot to load")
print("3. Then run the next cell")

## Step 6: Wait for Skills

Skills are the actions the robot can perform. The viewer must be open for skill discovery to work — the simulation runs in the browser.

This cell will keep retrying until your Chrome tab joins the room.

In [ ]:
async def wait_for_skills(timeout_s: float = 180.0):
    """Poll until the viewer joins and skills become available."""
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        try:
            found = await session.discover_skills()
            if found:
                return found
        except (RuntimeError, TimeoutError):
            pass  # viewer not joined yet
        await asyncio.sleep(2.0)
    raise TimeoutError("Viewer did not join — is the Chrome tab open?")


skills = await wait_for_skills()
print("Skills found:")
for skill in skills:
    print(f"  - {skill.name}")
    print(skill)

## Step 7: Read Robot Status

Before sending any commands, let's check the robot's current state. The `robot_status` read returns the robot's position, orientation, and overall status.

In [ ]:
state = await session.state.get("robot_status")
print(f"Robot status : {state.robot.status}")
print(f"Robot entity : {state.robot.entity_id}")
if state.robot.pose:
    print(f"Position     : {state.robot.pose.position}")
    print(f"Yaw (deg)    : {state.robot.pose.yaw_deg}")
print(f"Runtime ready: {state.runtime.status}")
print(f"Accepts cmds : {state.runtime.accepts_tool_calls}")

## Step 8: Read Scene State

Besides the robot's own status, you can read the full **scene state** — every entity in the warehouse (pads, cubes, the robot itself) along with their positions and properties.

This is scene state is not available on an actual robot. This is a simulation-only feature that is used for us to mock out features that the robot might have in the future. If your script relies on any information from the scene state, it means that the script will not be able to port to the real robot. 

The scene contains three kinds of entities:

| Entity type | Examples | What it is |
|---|---|---|
| **Robot** | `robot` | The humanoid itself — its position, orientation, and what it's holding |
| **Pads** | `pad_A`, `pad_B`, ... | Zones on the warehouse floor. `pad_A` is the conveyor belt that feeds in cubes; the others are color-coded delivery pads (much more on this in Section 7!) |
| **Cubes** | `cube_0`, `cube_1`, ... | Colored objects on the conveyor — these can be picked up and delivered |

(You may also notice some `cube_pool_*` entities with `visible=False` parked far away at (+20, +20) — that's the simulator's reserve supply. The belt pulls cubes from this hidden pool as you clear them, so the warehouse never runs out. Filtering on `visible` hides them.)

In [ ]:
scene = await session.state.get("scene_state")

# Sort entities into categories
robot_entities = []
pad_entities = []
cube_entities = []

for eid, entity in scene.entities.items():
    if eid == "robot":
        robot_entities.append((eid, entity))
    elif eid.startswith("pad_"):
        pad_entities.append((eid, entity))
    elif eid.startswith("cube") and entity.visible:
        cube_entities.append((eid, entity))
    # skip invisible pool cubes and duplicate aliases

# --- Robot ---
print("=== Robot ===")
for eid, e in robot_entities:
    pos = e.pose.position
    print(f"  {eid:12s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f})  yaw={e.pose.yaw_deg:.1f}\u00b0")

# --- Pads ---
print(f"\n=== Pads ({len(pad_entities)}) ===")
for eid, e in sorted(pad_entities):
    pos = e.pose.position
    print(f"  {eid:12s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f})")

# --- Visible cubes ---
print(f"\n=== Cubes ({len(cube_entities)}) ===")
for eid, e in sorted(cube_entities):
    pos = e.pose.position
    color = e.state.get("color", "?") if e.state else "?"
    pad = e.state.get("parent_pad_id", "?") if e.state else "?"
    held = f"  [held by {e.attached_to}]" if e.attached_to else ""
    print(f"  {eid:12s}  color={color:6s}  on={pad:5s}  pos=({pos[0]:+.2f}, {pos[1]:+.2f}, {pos[2]:+.2f}){held}")

---

# Section 5: Basic Usage of the Humanoid Policy Viewer

## Getting the Viewer

If you haven't already, open the viewer URL from Step 5 in a new Chrome tab. You should see a 3D warehouse scene with a humanoid robot standing on a platform.

The viewer is where the simulation actually runs — your notebook sends commands over the network, and the browser executes them in real time.

## Camera Controls

In the 3D viewer you can move the camera to get different perspectives of the scene:

| Action | How |
|---|---|
| **Rotate** | Click and drag (left mouse button) |
| **Pan** | Right-click and drag, or Shift + click and drag |
| **Zoom** | Scroll wheel |
| **Reset camera** | Press `R` to snap back to the default view |

Try moving the camera around now. Get familiar with the warehouse layout — you'll need to understand where things are to navigate the robot effectively.

## The Asimov Policy Viewer

The viewer is more than just a camera — it's the **runtime** for the `asimov-v0` policy. That means:

- The robot's physics, collision detection, and navigation all run **in your browser tab**.
- If you close the Chrome tab, the simulation stops and the SDK will lose its connection to the runtime.
- The viewer supports several interaction modes that let you control the robot directly (without the SDK).

### Click-to-Walk

**Try it now:** Click on the ground somewhere in the warehouse. The robot should walk to where you clicked.

This is the simplest way to move the robot. Behind the scenes, the viewer is running the same `go_to` skill that we'll use from the SDK.

### Auto-Nav

**Try it now:** Click on one of the labeled pads (e.g., `pad_A`, `pad_B`, `pad_C`). The robot will plan a path and walk there, navigating around obstacles.

Notice how the robot avoids walls and objects. This pathfinding is what makes `go_to` more powerful than raw velocity commands.

### Manual Velocity Commands

**Try it now:** Use the arrow keys to move the robot manually:

| Key | Action |
|---|---|
| `W` / `Up` | Walk forward |
| `S` / `Down` | Walk backward |
| `A` / `Left` | Turn left |
| `D` / `Right` | Turn right |

This is the manual equivalent of the `set_velocity` SDK command. The robot moves as long as you hold the key.

## How Do You Tell if Your Robot Has Fallen Over?

In the viewer, a fallen robot will be lying on its side or back — visually obvious. But from the SDK, you need to check programmatically.

The `robot_status` read includes a `status` field. The possible values are:

| Status | Meaning |
|---|---|
| `ready` | Idle, waiting for commands |
| `moving` | Currently executing a movement |
| `holding` | Holding an object |
| **`fallen`** | **The robot has fallen over** |
| `error` | Something went wrong |
| `unknown` | State not yet determined |

Run the cell below to check. If `status` shows `fallen`, you'll need to reset the scene (see troubleshooting at the bottom).

In [ ]:
state = await session.state.get("robot_status")
print(f"Robot status: {state.robot.status}")

if state.robot.status == "fallen":
    print("\nThe robot has fallen over! You may need to reset the scene.")
else:
    print("\nRobot is upright and operational.")

---

# Section 6: Writing a Simple Chain of Commands Using the Menlo SDK

In this section you'll write short programs that chain together SDK commands. This is how you'll eventually build more complex robot behaviors.

We'll use a helper function to take and display screenshots inline:

In [ ]:
from IPython.display import Image, display

async def screenshot(label: str = ""):
    """Grab the robot's POV camera and display it in the notebook."""
    jpeg = await session.get_vision("pov")
    if label:
        print(label)
    display(Image(data=jpeg, format="jpeg"))

## Before Exercise 1: Aiming the Camera with `set_head`

The robot can aim its head/neck independently of walking. This changes the POV camera direction, so it is useful for scanning the warehouse before deciding where to move.

`yaw` pans left/right and `pitch` tilts up/down. The command sets a target angle, and `robot_status` reports both the target and the measured head state.

In [ ]:
async def aim_head(yaw: float = 0.0, pitch: float = 0.0):
    """Aim the robot head/neck, then print target and measured state."""
    result = await session.invoke("set_head", {"yaw": yaw, "pitch": pitch})
    print(f"set_head -> {result.status}")

    state = await session.state.get("robot_status")
    head = state.robot.extra.get("head", {})
    print("target  :", head.get("target"))
    print("measured:", head.get("measured"))
    print("ranges  :", head.get("yaw_range"), head.get("pitch_range"))


# Try a short scan: center, look down slightly, look left, look right, then return center.
for label, yaw, pitch in [
    ("center", 0.0, 0.0),
    ("slightly down", 0.0, 0.15),
    ("left", 0.6, 0.15),
    ("right", -0.6, 0.15),
    ("center again", 0.0, 0.0),
]:

    print(f"\nHead pose: {label}")
    await aim_head(yaw=yaw, pitch=pitch)
    await screenshot(f"POV after aiming {label}:")

## Exercise 1: Simple Walking

Let's do the most basic sequence: take a screenshot, move the robot forward, then take another screenshot.

This lets you see the world from the robot's perspective and understand how it changes as the robot moves.

In [ ]:
# 1. Take a screenshot before moving
await screenshot("BEFORE — robot's point of view:")

In [ ]:
# 2. Move forward for 2 seconds
result = await session.invoke("set_velocity", {"vx": 0.8, "duration_s": 2.0})
print(f"set_velocity -> {result.status}")

In [ ]:
# 3. Take a screenshot after moving
await screenshot("AFTER — robot's point of view:")

### Guiding Questions

Look at the two screenshots above and think about these questions before moving on:

1. **Has the view changed?**
   Compare the before and after images carefully. What's different?

2. **Why did the view change?**
   Can you estimate *how far* the robot moved? (Hint: look at the numbers you passed to `set_velocity` — a speed multiplied by a time gives a distance.)

3. **How is the robot's perspective different from the one you've been looking at so far?**
   You now have two ways of seeing the world: the 3D viewer in Chrome, and the robot's POV camera. What can you see in one that you can't see in the other?

4. **What are some difficulties you foresee when working with the limitations of the sensors available on the robot?**
   Imagine you could *only* use the POV camera (no 3D viewer). What tasks would become harder?

## Before Exercise 2: How to Turn by a Specific Angle

Exercise 2 will require turning the robot by roughly 90°. There's no "turn 90 degrees" command — `set_velocity` only takes a *rotation speed* and a *duration*. So we need a little math, plus one quirk of walking robots.

### Radians, not degrees

The rotation speed `wz` is measured in **radians per second**. Radians are just another unit for angles:

| Degrees | Radians (exact) | Radians (approx.) |
|---|---|---|
| 90° | π/2 | 1.57 |
| 180° | π | 3.14 |
| 360° | 2π | 6.28 |

### Speed × time = angle

Exactly like distance = speed × time for walking, the angle turned is:

**angle turned (radians) = `wz` × `duration_s`**

### Quirk #1: this robot turns like a bicycle

Try sending `{"wz": 0.5, "duration_s": 3.14}` on its own and… almost nothing happens. The locomotion policy that drives the legs **cannot spin in place** — just like a bicycle, it can only turn *while moving forward*. Always combine `wz` with a small forward speed:

```python
{"wz": 0.5, "vx": 0.2, "duration_s": 3.14}   # turns properly
{"wz": 0.5, "duration_s": 3.14}              # barely turns at all!
```

The small `vx` means the robot traces a gentle arc (a few tens of centimeters) instead of pivoting on the spot — plan a little extra room for it.

### Quirk #2: speed limits (silent clipping)

The policy was only trained up to certain speeds, so the simulator **clips** anything faster *without telling you*:

| Parameter | Meaning | Limit |
|---|---|---|
| `vx` | forward (+) / backward (−), m/s | ±1.5 |
| `vy` | left (+) / right (−) sidestep, m/s | ±1.5 |
| `wz` | turn left (+) / turn right (−), rad/s | **±0.6** |

If you ask for `wz = 1.0`, the robot actually turns at 0.6 rad/s — and your carefully computed angle will be wrong. **Keep `wz` at 0.6 or less so the math you do matches what the robot does.**

### Recipes you can reuse

```python
# Turn LEFT ~90°:  0.5 rad/s × 3.14 s ≈ 1.57 rad ≈ 90°  (vx makes the turn work)
await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 3.14})

# Turn RIGHT ~90°: negative wz turns the other way
await session.invoke("set_velocity", {"wz": -0.5, "vx": 0.2, "duration_s": 3.14})

# Turn LEFT ~180°: same speed, twice the time
await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 6.28})
```

### Expect some error

The math gives a *target*, not a guarantee. Measured on the real sim, a commanded 90° turn lands anywhere from ~75° to ~100° — the policy ramps up gradually, wobbles as it steps, and doesn't stop instantly. You can always check the real heading afterwards:

```python
state = await session.state.get("robot_status")
print(state.robot.pose.yaw_deg)   # heading in degrees: 0° = +x direction, 90° = +y
```

Checking the actual result and correcting is a big idea in robotics — we'll come back to it at the end of Exercise 2.

## Exercise 2: Going Around the Wall

Now for a harder challenge. Using the third-person view in the Chrome viewer, identify a wall or obstacle in the warehouse. (The conveyor belt right in front of the robot's spawn point is a perfect one — it blocks the straight path to the area behind it.)

**Your task:** Write a script that navigates the robot around the obstacle using only `set_velocity` commands — no `go_to` allowed! (`go_to` has a built-in path planner that would do all the work for you. The point of this exercise is to feel what life is like *without* one.)

You have three velocity parameters:

| Parameter | Meaning | Limit |
|---|---|---|
| `vx` | forward (+) / backward (−) speed in m/s | ±1.5 |
| `vy` | left (+) / right (−) sidestep speed in m/s | ±1.5 |
| `wz` | rotation speed in rad/s, positive = turn left, negative = turn right | ±0.6 |

You can combine them in one command — and for `wz` you *must* (turns need a little `vx`, remember the primer).

A basic strategy might look like:
1. Turn ~90 degrees so you're facing along the wall
2. Walk forward until you're past the wall's end
3. Turn back ~90 degrees the other way
4. Walk forward — you're now past the wall
5. (Optional) turn and walk to end up *behind* it

You could also skip turning entirely and use `vy` to sidestep along the wall — both are valid solutions. Either way: take a screenshot and read the robot's position between commands to check your progress. Here's a skeleton to get you started — fill in the values:

In [ ]:
# A small helper we'll reuse: print the robot's position and heading
async def print_position(label: str = ""):
    state = await session.state.get("robot_status")
    p = state.robot.pose.position
    print(f"{label:20s} pos=({p[0]:+.2f}, {p[1]:+.2f})  yaw={state.robot.pose.yaw_deg:+.1f}°  status={state.robot.status}")

# Take a screenshot to see where we are
await screenshot("Starting position:")
await print_position("START")

In [ ]:
# ============================================================
# YOUR CODE HERE
# Write a sequence of set_velocity commands to navigate
# the robot around a wall.
#
# Example commands:
#   await session.invoke("set_velocity", {"vx": 0.8, "duration_s": 3.0})               # forward ~2 m
#   await session.invoke("set_velocity", {"wz": 0.5, "vx": 0.2, "duration_s": 3.14})   # turn left ~90°
#   await session.invoke("set_velocity", {"wz": -0.5, "vx": 0.2, "duration_s": 3.14})  # turn right ~90°
#
# Remember:
#   * turns need the small vx — this robot can't spin in place!
#   * |vx|, |vy| ≤ 1.5 and |wz| ≤ 0.6 — faster values are silently clipped
#
# Tip: check your progress between moves:
#   await screenshot("After turning:")
#   await print_position("after turn")
# ============================================================



In [ ]:
# Check where we ended up
await screenshot("Final position:")

state = await session.state.get("robot_status")
print(f"Final position: {state.robot.pose.position}")
print(f"Final yaw: {state.robot.pose.yaw_deg} degrees")
print(f"Robot status: {state.robot.status}")

### Guiding Questions

After you've gotten your script to work (or tried a few times), think about these questions:

1. **Does the script work every single time?**
   Run your script 2–3 times (you may need to reset the robot's position first — `go_to` back to where you started, or use the viewer's click-to-walk). Does the robot end up in *exactly* the same place each run? Why or why not?

2. **What would happen if the position of the wall changed slightly?**
   Say the wall moved half a meter. Would your script still work? What does that tell you about this style of controlling a robot?

3. **What information would your script need to handle a moved wall, and where could it come from?**
   Look back at the tools you've already used in this notebook — is there anything that could tell the robot where it is, or what's in front of it?

---

# Section 7: Pick and Place

Walking is nice, but warehouses are about *moving things*. In this section the robot will pick up a cube from the conveyor belt and deliver it to a pad — the classic **pick and place** task that underlies almost all warehouse robotics.

### The sorting game

This warehouse is actually a little game. The conveyor (`pad_A`) feeds in colored cubes, and each delivery pad only **accepts one color** — look at the pad signs in the viewer, they're color-coded:

| Color | Delivery pad |
|---|---|
| 🟩 green | `pad_C` |
| 🟦 blue | `pad_D` |
| 🟥 red | `pad_B` |
| 🟨 yellow | `pad_E` |

Deliver a cube to its matching pad and the warehouse *accepts* it — the cube disappears from the scene, and the belt feeds in a new one.

> ⚠️ **Deliver to the wrong pad and the sorting benchmark ends.** After that, every `pick_entity` fails with `benchmark terminated: wrong_color_destination`. If that happens, re-arm it with:
> ```python
> await session.invoke("configure_benchmark", {"benchmark": {}})
> ```

### The two new skills

| Skill | What it does | Precondition |
|---|---|---|
| `pick_entity` | Grasp a cube | Robot must be **within reach** (~1 m), hands empty |
| `place_entity` | Deliver the held cube to a pad | Robot must be holding something |

Both take the same **target format** you've seen with `go_to`:

```python
{"target": {"kind": "entity", "entity_id": "<name>"}}
```

Two ways to pick:
- **By name** — `"entity_id": "cube_4"` grasps that exact cube (it must be within reach).
- **Nearest** — the special name `"cube"` means *"whichever cube is closest"*. Handy at the front of the belt, but risky if two cubes are near each other — you might grab the wrong one.

Either way, the recipe is the same: **position first, then pick** — walk next to the cube you want before grasping.

## Step 1: Survey the Scene

First, find out where everything is *right now* — your robot has moved around since Section 4, and the conveyor keeps feeding cubes. We read `scene_state` for exact positions and take a POV screenshot to see the world from the robot's eyes.

Note: Reading the scene state is not possible on the actual robot. For the purposes of this tutorial, we will use it first to simplify how the code works. 

In [ ]:
# Where is the robot, and where are the cubes?
scene = await session.state.get("scene_state")

robot = scene.entities["robot"]
rp = robot.pose.position
print(f"Robot at ({rp[0]:+.2f}, {rp[1]:+.2f}), yaw={robot.pose.yaw_deg:.0f}°")

print("\nCubes (sorted by distance from the robot):")
cubes = []
for eid, e in scene.entities.items():
    if eid.startswith("cube_") and e.visible:
        p = e.pose.position
        dist = ((p[0] - rp[0]) ** 2 + (p[1] - rp[1]) ** 2) ** 0.5
        color = e.state.get("color", "?") if e.state else "?"
        cubes.append((dist, eid, color, p))

for dist, eid, color, p in sorted(cubes):
    print(f"  {eid:8s} color={color:6s} at ({p[0]:+.2f}, {p[1]:+.2f})  -> {dist:.2f} m away")

await screenshot("\nRobot's POV:")

## Step 2: Get Within Reach

`pick_entity` only works if the robot is standing close enough to a cube. So before picking, we **position** the robot next to one — we'll `go_to` the nearest cube found in our survey.

> **Heads-up — a common trap:** the robot does *not* spawn exactly in front of the conveyor. Right after loading the scene it stands slightly to the side of the first box, just out of grasping range — so calling `pick_entity` as your very first command fails, even though the belt *looks* close in the viewer. "Looks close" and "is within reach" are different things, which is exactly why we position first and measure the distance with `scene_state` instead of eyeballing it. (A small sidestep — `set_velocity` with a negative `vy` for about a second — is enough to fix it at spawn; the `go_to` we use here handles it in general.)

In [ ]:
# Walk to the nearest cube so it's within reach
nearest_dist, nearest_cube, _, _ = min(cubes)
print(f"Nearest cube: {nearest_cube} ({nearest_dist:.2f} m away) — walking to it...")

result = await session.invoke(
    "go_to", {"target": {"kind": "entity", "entity_id": nearest_cube}}, timeout_s=300
)
print(f"go_to {nearest_cube} -> {result.status}")

# How close are we now?
scene = await session.state.get("scene_state")
rp = scene.entities["robot"].pose.position
cp = scene.entities[nearest_cube].pose.position
dist = ((cp[0] - rp[0]) ** 2 + (cp[1] - rp[1]) ** 2) ** 0.5
print(f"Distance to {nearest_cube} is now {dist:.2f} m")

await screenshot("In position:")

## Step 3: Pick!

Now that we're standing next to a cube, we can grasp it. Here we use the `"cube"` target — *the nearest cube* — and then immediately check the scene state to confirm what we're holding. A held cube shows up with its `attached_to` field set, and the robot's own status changes to `holding`.

Why check? Because "nearest" is decided **at pick time**, and the belt keeps moving — it's entirely possible to walk toward one cube and end up grasping its neighbor. (We saw it happen during testing!) Our Step 4 code handles that gracefully: it reads the color of whatever we *actually* grabbed and delivers it to the right pad. When you need one *specific* cube, pick it by its exact `entity_id` instead — you'll do that in the exercise.

In [ ]:
# Pick up the nearest cube ("cube" = whichever one is closest)
result = await session.invoke(
    "pick_entity", {"target": {"kind": "entity", "entity_id": "cube"}}, timeout_s=300
)
print(f"pick_entity -> {result.status}")

# Confirm the grab in the scene state: which cube is attached to the robot?
scene = await session.state.get("scene_state")
held = [
    (eid, e.state.get("color", "?") if e.state else "?")
    for eid, e in scene.entities.items()
    if eid.startswith("cube_") and e.attached_to
]
print(f"Now holding: {held}")

state = await session.state.get("robot_status")
print(f"Robot status: {state.robot.status}")   # should say 'holding'

await screenshot("After picking:")

## Step 4: Carry It to the Right Pad and Deliver It

The robot is holding a cube — it will carry it automatically while walking. But remember the sorting rule: the cube must go to **the pad that matches its color**, or the benchmark ends. So the code below:

1. checks **what color** we're actually holding (from `scene_state`),
2. looks up the matching pad in a little dictionary,
3. uses `go_to` (the path planner) to walk there,
4. delivers with `place_entity`,
5. and **verifies the delivery** — an accepted cube vanishes from the scene (`visible` becomes `False`) and the robot goes back to `ready`.

That step 5 is the interesting one: success here means the cube *disappears* (the warehouse accepted the parcel), so the only way to confirm it is by reading the world's state — your eyes have nothing to look at!

(One more detail: `place_entity`'s target is optional — omit it and the robot delivers to the **nearest** known zone. We always pass the pad explicitly, because "nearest" plus a color rule is how accidents happen.)

In [ ]:
# Which pad accepts which color (check the pad signs in the viewer!)
COLOR_TO_PAD = {"green": "pad_C", "blue": "pad_D", "red": "pad_B", "yellow": "pad_E"}

# 1-2. What are we holding, and where must it go?
scene = await session.state.get("scene_state")
held_cube, held_color = next(
    (eid, e.state.get("color"))
    for eid, e in scene.entities.items()
    if eid.startswith("cube_") and e.attached_to
)
target_pad = COLOR_TO_PAD[held_color]
print(f"Holding {held_cube} ({held_color}) -> delivering to {target_pad}")

# 3. Walk there with the path planner
result = await session.invoke(
    "go_to", {"target": {"kind": "entity", "entity_id": target_pad}}, timeout_s=300
)
print(f"go_to {target_pad} -> {result.status}")

# 4. Deliver
result = await session.invoke(
    "place_entity", {"target": {"kind": "entity", "entity_id": target_pad}}, timeout_s=300
)
print(f"place_entity -> {result.status}")

# 5. Verify with DATA, not eyes: the delivered cube should be GONE,
#    and the robot should no longer be holding anything.
scene = await session.state.get("scene_state")
state = await session.state.get("robot_status")
print(f"{held_cube} still visible? {scene.entities[held_cube].visible}")
print(f"Robot status: {state.robot.status}")          # back to 'ready'
delivered = not scene.entities[held_cube].visible and state.robot.status != "holding"
print(f"Delivery confirmed: {delivered}")

await screenshot("After delivering:")

### Guiding Questions

1. **What do you think happens if you call `pick_entity` while the robot is far away from every cube?** Make a prediction, then try it (walk somewhere empty first). What does the `result` look like?

2. **What happens if you call `pick_entity` while the robot is already holding a cube?** (Look back at the skill's description in Section 4, Step 6 — it already tells you.)

3. **What happens if you deliver a cube to the wrong-color pad?** (Careful — actually trying this one has consequences. Read the warning at the top of this section first.)

4. **After delivering, we checked the cube's `visible` flag and the robot's status in `scene_state` instead of just looking at the viewer or trusting `result.status`. Why is that the better habit?**

## Exercise: Deliver a Red Cube

Your turn. **Write a program that finds a *red* cube on the conveyor and delivers it to the correct pad.**

This combines everything from this section:

- Use `scene_state` to *find* a red cube (don't hardcode a cube name — the belt moves and refills, so the ids change between runs!).
- Walk to it, pick it — and **check what you're actually holding**. Is it really red? (If you used the `"cube"` nearest-pick, what could go wrong here?)
- Deliver it to the right pad (look at the table at the top of this section — getting this wrong ends the benchmark!).
- **Verify in code** that the delivery was accepted. What happens to a cube when it's delivered?

One pitfall to watch for: if the robot stops *between* two cubes, "nearest" might not be the one you wanted. How can your code notice — and what's the safer way to pick?

In [ ]:
# ============================================================
# YOUR CODE HERE
#
# Plan:
#   1. Read scene_state and find a RED cube (check e.state["color"])
#   2. go_to that cube
#   3. pick_entity — by its exact entity_id (safer than "cube"!)
#   4. Check what you're holding (attached_to) — is it the red one?
#   5. go_to the pad that accepts red, then place_entity on it
#   6. Read scene_state again and PROVE the delivery:
#      the cube you held should now have visible == False
#
# Useful snippets:
#   await session.invoke("go_to", {"target": {"kind": "entity", "entity_id": "..."}}, timeout_s=300)
#   color = e.state.get("color") if e.state else None
# ============================================================



---
## Cleanup

In [ ]:
await session.disconnect()
await client.robots.delete(robot_id)
print("Robot deleted.")